In [8]:
# ==========================================
# ENVIRONMENT SETUP
# ==========================================
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

def find_project_root(markers=(".git", "pyproject.toml", "src")):
    """
    Ascending tree search to locate project root.
    """
    current = Path.cwd()
    for parent in [current] + list(current.parents):
        if any((parent / marker).exists() for marker in markers):
            return parent
    raise RuntimeError("Project root not found.")

project_root = str(find_project_root())

if project_root not in sys.path:
    sys.path.insert(0, project_root)



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
# ============================================================
# IMPORT EXPERIMENT
# ============================================================

from experiments.scripts.EXP_001_MP_H0_MONTECARLO_VALIDATION import exp_001_mp_h0_montecarlo

In [10]:
# ==========================================
# EXPERIMENT CONFIGURATION
# ==========================================
CONFIG = {
    "n": 500,
    "p": 300,
    "M": 200,
    "seed": 42,
}

In [11]:
# ==========================================
# RUN EXPERIMENT
# ==========================================
output = exp_001_mp_h0_montecarlo(**CONFIG)

results = output["results"]
meta = output["meta"]

results

{'mean_k': 0.13,
 'std_k': 0.3508560958569767,
 'mean_ratio': 0.9819461240213718,
 'mean_sigma2': 0.9990529056342686}

In [12]:
# ==========================================
# METADATA
# ==========================================
print("\n=== META ===")
for k, v in meta.items():
    print(f"{k}: {v}")


=== META ===
n: 500
p: 300
M: 200
seed: 42
execution_time_minutes: 0.05
model_version: 0.1.0


In [13]:
# ==========================================
# QUICK DIAGNOSTICS
# ==========================================
print("=== H0 VALIDATION RESULTS ===")
print(f"Mean k (False Positives): {results['mean_k']:.4f}")
print(f"Std k: {results['std_k']:.4f}")
print(f"Mean Apex Ratio: {results['mean_ratio']:.4f}")
print(f"Estimated sigma^2: {results['mean_sigma2']:.4f}")

=== H0 VALIDATION RESULTS ===
Mean k (False Positives): 0.1300
Std k: 0.3509
Mean Apex Ratio: 0.9819
Estimated sigma^2: 0.9991


In [14]:
# ==========================================
# THEORY CHECK
# ==========================================
import numpy as np

print("\n=== THEORY CHECK ===")

# Bajo H0:
# sigma^2 debería ser ≈ 1
# k_effective debería ser ≈ 0 (pero no exactamente por TW)

print("Expected sigma^2 ≈ 1")
print("Observed sigma^2:", results["mean_sigma2"])

print("\nExpected k ≈ 0 (finite sample noise allowed)")
print("Observed mean_k:", results["mean_k"])


=== THEORY CHECK ===
Expected sigma^2 ≈ 1
Observed sigma^2: 0.9990529056342686

Expected k ≈ 0 (finite sample noise allowed)
Observed mean_k: 0.13


### Interpretation

- **Does it match theory?**
  Yes. The bulk variance estimator ($\sigma^2$) perfectly recovers the theoretical unit variance of the Wishart null hypothesis (Observed: 0.9991 vs Expected: 1.0). The Mean Apex Ratio (0.9819) confirms the empirical spectral mass is structurally aligned with the theoretical Marchenko-Pastur density limit.

- **Any bias or finite-sample effects?**
  A known finite-sample edge effect is present. The mean false positive count ($k \approx 0.13$) is strictly greater than zero. This occurs because, in finite dimensions ($N=500, P=300$), the Tracy-Widom fluctuations of the largest sample eigenvalues naturally leak past the rigid asymptotic boundary ($\lambda_+$).

- **Pipeline Implications:**
  This experiment establishes the baseline Type I error of the analytic engine. It empirically proves why the standard Marchenko-Pastur threshold is insufficient for perfect statistical power, structurally justifying the necessity of the Bootstrap module for exact calibration in production.